# 02b — Spark Streaming Consumer: Silver Layer (2026 Live)

## 🎯 Architecture — Kappa Hybrid

This notebook is the **real-time processing engine** of the pipeline.
It reads from Kafka using `spark.readStream`, processes each micro-batch via
`foreachBatch`, and writes results to MongoDB — all without blocking the cluster.

```
[Kafka: tlc-yellow-2026]
        │
  readStream (Kafka source)
        │
  foreachBatch(process_batch)
     ├── apply_quality_flags()       ← flag-based, no silent drops
     ├── route_quarantine()          ← split valid / rejected
     ├── enrich_trip_locations()     ← broadcast zone join
     ├── build_yellow_silver()       ← Silver schema + bronze lineage
     ├── write → tlc_silver.trips_yellow
     ├── write → tlc_audit.quarantine
     └── log → tlc_audit.pipeline_runs (streaming batch audit)
```

## ⚠️ Critical Streaming Rules (never break these)

1. **`.count()` is FORBIDDEN on the streaming DataFrame** (`df_stream`).  
   All counts happen inside `foreachBatch`, where `batch_df` is a **static** micro-batch.
2. **`checkpointLocation` is mandatory** — enables exactly-once delivery and safe restarts.
3. **Start this notebook BEFORE the Producer** (`01b`) — the consumer must be listening.

---
> **Run order**: Run ALL cells top to bottom. The stream starts in the last cell.
> The stream runs indefinitely — stop the query manually when done.

## Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

import os
import json
import datetime
import uuid
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

from src.spark_utils import get_spark_streaming, write_mongo
from src.transformations.tlc_rules import (
    YELLOW_GREEN_RULES, apply_quality_flags, route_quarantine
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_yellow_silver
from core.config.settings import settings
from core.storage.mongo_client import get_audit_db

# ── Kafka configuration ───────────────────────────────────────────────────────
KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
KAFKA_TOPIC     = os.getenv("KAFKA_TOPIC_YELLOW_2026",  "tlc-yellow-2026")
VEHICLE_TYPE    = "yellow"

# ── Checkpoint — critical for exactly-once and safe restarts ──────────────────
# This path persists across notebook restarts.
# Delete this folder only if you want to reprocess from the beginning.
CHECKPOINT_BASE = "/home/jovyan/work/data/checkpoints"
CHECKPOINT_PATH = f"{CHECKPOINT_BASE}/silver_stream_{VEHICLE_TYPE}_2026"

# ── Streaming trigger ─────────────────────────────────────────────────────────
# processingTime: how often Spark polls Kafka for new records
TRIGGER_INTERVAL = "30 seconds"   # adjust to "5 seconds" for faster demo

# ── Kafka read settings ───────────────────────────────────────────────────────
STARTING_OFFSETS = "earliest"   # process from start; use "latest" after 1st run
MAX_OFFSETS_PER_TRIGGER = 5000  # max records per micro-batch (controls memory)

print(f"Kafka bootstrap  : {KAFKA_BOOTSTRAP}")
print(f"Kafka topic      : {KAFKA_TOPIC}")
print(f"Checkpoint path  : {CHECKPOINT_PATH}")
print(f"Trigger interval : {TRIGGER_INTERVAL}")
print(f"Max offsets/batch: {MAX_OFFSETS_PER_TRIGGER:,}")

# Create checkpoint directory
Path(CHECKPOINT_PATH).mkdir(parents=True, exist_ok=True)
print(f"\n✓ Checkpoint directory ready.")

## Start Spark Streaming Session

Uses `get_spark_streaming()` which loads both the **MongoDB** and **Kafka** connectors.

In [ ]:
spark = get_spark_streaming(
    app_name="TLC_Silver_Stream_2026",
    kafka_bootstrap=KAFKA_BOOTSTRAP,
)
print(f"Spark version: {spark.version}")
print(f"Spark UI: http://localhost:4060")

## Define JSON Schema for Kafka Messages

Kafka delivers each message as raw bytes. We define the expected schema
to parse the JSON payload produced by notebook `01b`.

> Tip: This schema mirrors the Yellow Taxi Parquet schema plus the `_stream_meta` field injected by the Producer.

In [ ]:
# Schema for the _stream_meta envelope injected by the Producer
STREAM_META_SCHEMA = T.StructType([
    T.StructField("produced_at",  T.StringType(),  True),
    T.StructField("source_file",  T.StringType(),  True),
    T.StructField("vehicle_type", T.StringType(),  True),
    T.StructField("batch_id",     T.IntegerType(), True),
])

# Schema for Yellow Taxi trip record fields
YELLOW_RECORD_SCHEMA = T.StructType([
    T.StructField("VendorID",               T.IntegerType(), True),
    T.StructField("tpep_pickup_datetime",   T.StringType(),  True),   # ISO string from JSON
    T.StructField("tpep_dropoff_datetime",  T.StringType(),  True),
    T.StructField("passenger_count",        T.DoubleType(),  True),
    T.StructField("trip_distance",          T.DoubleType(),  True),
    T.StructField("RatecodeID",             T.DoubleType(),  True),
    T.StructField("store_and_fwd_flag",     T.StringType(),  True),
    T.StructField("PULocationID",           T.IntegerType(), True),
    T.StructField("DOLocationID",           T.IntegerType(), True),
    T.StructField("payment_type",           T.IntegerType(), True),
    T.StructField("fare_amount",            T.DoubleType(),  True),
    T.StructField("extra",                  T.DoubleType(),  True),
    T.StructField("mta_tax",               T.DoubleType(),  True),
    T.StructField("tip_amount",             T.DoubleType(),  True),
    T.StructField("tolls_amount",           T.DoubleType(),  True),
    T.StructField("improvement_surcharge",  T.DoubleType(),  True),
    T.StructField("total_amount",           T.DoubleType(),  True),
    T.StructField("congestion_surcharge",   T.DoubleType(),  True),
    T.StructField("airport_fee",            T.DoubleType(),  True),
    T.StructField("cbd_congestion_fee",     T.DoubleType(),  True),
    T.StructField("_stream_meta",           STREAM_META_SCHEMA, True),
])

print("JSON schema defined.")

## Pre-load Zone Lookup (Broadcast)

We load the zone lookup **once** before the streaming loop starts.
The broadcast join in `enrich_trip_locations()` will cache it efficiently across batches.

In [ ]:
zone_df_broadcast = load_zone_lookup(spark)
# Cache the zone lookup — it's tiny (265 rows) and used in every batch
zone_df_broadcast.cache()
zone_count = zone_df_broadcast.count()
print(f"✓ Zone lookup cached: {zone_count} zones")

## Define `foreachBatch` Processing Function

This is the **heart** of the consumer. Every 30 seconds Spark delivers a **static**
micro-batch DataFrame (`batch_df`) — safe to call `.count()`, `.filter()`, and `.write()` on.

Processing steps per batch:
1. Parse Kafka JSON payload → typed DataFrame
2. Cast datetime strings → timestamps
3. Inject Bronze-equivalent `_meta` from `_stream_meta`
4. Apply quality flags (`apply_quality_flags`)
5. Route: valid → Silver, rejected → Quarantine
6. Enrich valid records with zone metadata
7. Build Silver schema (`build_yellow_silver`)
8. Write Silver + Quarantine to MongoDB
9. Log audit record to `tlc_audit.pipeline_runs`

In [ ]:
# Shared batch counter (mutable via list trick)
_batch_stats = {"total_silver": 0, "total_quarantine": 0, "batches_processed": 0}


def process_batch(batch_df: DataFrame, batch_id: int) -> None:
    """
    foreachBatch callback — receives a STATIC micro-batch DataFrame.

    This function is called by Spark Structured Streaming every trigger interval.
    All .count(), .filter(), and .write() operations are SAFE here.

    Parameters
    ----------
    batch_df:
        Raw Kafka micro-batch. Schema: [key, value, topic, partition, offset, timestamp, ...]
    batch_id:
        Monotonically increasing batch sequence number (for idempotency).
    """
    batch_start = datetime.datetime.utcnow()
    stream_run_id = f"stream_{batch_id:06d}_{str(uuid.uuid4())[:8]}"

    print(f"\n── Batch {batch_id} | run_id={stream_run_id} | {batch_start.isoformat()} ──")

    # ── Step 1: Skip empty batches ─────────────────────────────────────────────
    if batch_df.rdd.isEmpty():
        print(f"  [Batch {batch_id}] Empty — no records from Kafka. Waiting...")
        return

    # ── Step 2: Parse Kafka value bytes → JSON → typed schema ──────────────────
    parsed_df = (
        batch_df
        .select(
            F.from_json(
                F.col("value").cast("string"),
                YELLOW_RECORD_SCHEMA
            ).alias("data"),
            F.col("partition").alias("_kafka_partition"),
            F.col("offset").alias("_kafka_offset"),
        )
        .select("data.*", "_kafka_partition", "_kafka_offset")
    )

    # Cast ISO datetime strings → Spark timestamps
    parsed_df = (
        parsed_df
        .withColumn("tpep_pickup_datetime",  F.to_timestamp("tpep_pickup_datetime"))
        .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
        .withColumn("passenger_count",       F.col("passenger_count").cast("int"))
    )

    # ── Step 3: Inject Bronze-equivalent _meta from _stream_meta ───────────────
    # In streaming, Bronze is bypassed. We attach equivalent lineage metadata
    # using the Producer's _stream_meta envelope.
    parsed_df = parsed_df.withColumn(
        "_meta",
        F.struct(
            F.col("_stream_meta.vehicle_type").alias("vehicle_type"),
            F.lit(stream_run_id).alias("run_id"),          # this IS the bronze_run_id
            F.current_timestamp().alias("ingestion_time"),
            F.col("_stream_meta.source_file").alias("source_file"),
        )
    ).drop("_stream_meta")

    # Drop Kafka internal columns before quality checks
    parsed_df = parsed_df.drop("_kafka_partition", "_kafka_offset")

    # Count bronze-equivalent records (SAFE: batch_df is static)
    records_in = parsed_df.count()
    print(f"  Records received from Kafka: {records_in:,}")

    if records_in == 0:
        print(f"  [Batch {batch_id}] All records failed JSON parsing. Skipping.")
        return

    # ── Step 4: Apply quality flags ────────────────────────────────────────────
    flagged_df = apply_quality_flags(parsed_df, YELLOW_GREEN_RULES)

    # ── Step 5: Route valid → Silver, rejected → Quarantine ────────────────────
    valid_df, rejected_df = route_quarantine(flagged_df)

    # Count both partitions (SAFE in foreachBatch)
    records_valid    = valid_df.count()
    records_rejected = rejected_df.count()
    quar_rate        = records_rejected / records_in * 100 if records_in > 0 else 0

    print(f"  Valid     : {records_valid:,}")
    print(f"  Rejected  : {records_rejected:,}  ({quar_rate:.2f}% quarantine rate)")

    # ── Step 6: Enrich valid records with zone metadata ────────────────────────
    if records_valid > 0:
        enriched_df = enrich_trip_locations(valid_df, zone_df_broadcast)

        # ── Step 7: Build Silver schema ────────────────────────────────────────
        # stream_run_id acts as both the Silver run_id and the bronze_run_id
        # (since we bypass Bronze in streaming mode)
        silver_df = build_yellow_silver(enriched_df, run_id=stream_run_id)

        # ── Step 8a: Write Silver records → MongoDB ────────────────────────────
        n_silver = write_mongo(
            silver_df,
            settings.MONGO_DB_SILVER,
            "trips_yellow",
            mode="append",
        )
        print(f"  ✓ Silver written: {n_silver:,} rows → tlc_silver.trips_yellow")
        _batch_stats["total_silver"] += n_silver

    # ── Step 8b: Write Quarantine records → MongoDB ────────────────────────────
    if records_rejected > 0:
        audit_db = get_audit_db()
        quarantine_docs = [
            {
                "quarantined_at": datetime.datetime.utcnow(),
                "silver_run_id":  stream_run_id,
                "bronze_run_id":  stream_run_id,   # same in streaming mode
                "source_file":    row.asDict(recursive=True).get("_meta", {}).get("source_file"),
                "pipeline":       "silver_stream_yellow",
                "reason":         row["_rejection_reason"],
                "quality_flags":  row.asDict(recursive=True).get("quality_flags", {}),
                "kafka_batch_id": batch_id,
                "raw_record":     {
                    k: v for k, v in row.asDict().items()
                    if k not in ("_rejection_reason", "quality_flags")
                },
            }
            for row in rejected_df.limit(1000).collect()  # cap for safety
        ]
        if quarantine_docs:
            audit_db["quarantine"].insert_many(quarantine_docs)
            print(f"  ✓ Quarantine: {len(quarantine_docs):,} records → tlc_audit.quarantine")
            _batch_stats["total_quarantine"] += len(quarantine_docs)

    # ── Step 9: Log streaming batch audit record ───────────────────────────────
    batch_end       = datetime.datetime.utcnow()
    duration_s      = (batch_end - batch_start).total_seconds()
    throughput      = records_in / duration_s if duration_s > 0 else 0

    audit_record = {
        "execution_id":   stream_run_id,
        "pipeline_name":  "silver_stream_yellow",
        "status":         "SUCCESS",
        "kafka_batch_id": batch_id,
        "start_time":     batch_start.isoformat(),
        "end_time":       batch_end.isoformat(),
        "duration_seconds": round(duration_s, 2),
        "output_summary": {
            "records_read_from_bronze":    records_in,
            "records_written_to_silver":   records_valid,
            "records_quarantined":         records_rejected,
            "quarantine_rate_pct":         round(quar_rate, 4),
            "throughput_records_per_s":    round(throughput, 1),
        },
        "quality_checks": [],
        "errors": [],
        "mode": "streaming_kafka",
    }

    try:
        get_audit_db()["pipeline_runs"].insert_one(audit_record)
    except Exception as e:
        print(f"  ⚠ Audit log write failed: {e}")

    _batch_stats["batches_processed"] += 1

    # ── Batch summary ──────────────────────────────────────────────────────────
    print(f"  Duration  : {duration_s:.1f}s  |  Throughput: {throughput:,.0f} rec/s")
    print(f"  Cumulative: Silver={_batch_stats['total_silver']:,}  "
          f"Quarantine={_batch_stats['total_quarantine']:,}  "
          f"Batches={_batch_stats['batches_processed']}")


print("✓ foreachBatch function 'process_batch' defined.")

## Create Kafka readStream

This cell creates the **streaming DataFrame** — it does NOT start processing yet.
Processing starts in the next cell when `.start()` is called.

In [ ]:
# ⚠ df_stream is a STREAMING DataFrame — NEVER call .count(), .show(), .collect()
#   on this object outside of foreachBatch!

df_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", STARTING_OFFSETS)
    .option("maxOffsetsPerTrigger", MAX_OFFSETS_PER_TRIGGER)
    .option("kafka.group.id", "tlc_silver_consumer")
    # Fault tolerance: don't fail if topic doesn't exist yet
    .option("kafka.max.poll.records", "1000")
    .option("failOnDataLoss", "false")
    .load()
)

print(f"✓ Streaming DataFrame created from topic '{KAFKA_TOPIC}'")
print(f"  Schema: {df_stream.schema.simpleString()}")
print(f"  isStreaming: {df_stream.isStreaming}")

## 🚀 Start Streaming Query

The query runs **continuously** in the background.
Each trigger delivers a micro-batch to `process_batch()`.

**Exam Day Checklist:**
- ✅ Checkpoint path set → safe to restart without reprocessing
- ✅ `failOnDataLoss=false` → tolerant to Kafka topic unavailability  
- ✅ `maxOffsetsPerTrigger` limits memory per micro-batch
- ✅ All `.count()` calls are inside `foreachBatch` (static batch)

To **stop** the stream: run `streaming_query.stop()` in a new cell.

In [ ]:
streaming_query = (
    df_stream
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .queryName(f"tlc_silver_stream_{VEHICLE_TYPE}")
    .start()
)

print(f"✓ Streaming query started!")
print(f"  Query name  : {streaming_query.name}")
print(f"  Query ID    : {streaming_query.id}")
print(f"  Status      : {streaming_query.status}")
print(f"  Checkpoint  : {CHECKPOINT_PATH}")
print(f"")
print(f"  🟢 Waiting for records from Kafka topic '{KAFKA_TOPIC}'...")
print(f"  → Now run notebook 01b to start the Producer.")
print(f"  → Monitor at Spark UI: http://localhost:4060")
print(f"  → Monitor at Kafka UI: http://localhost:8090")
print(f"")
print(f"  To stop the stream: streaming_query.stop()")

## Monitor Stream Progress

Run this cell at any time to check the live status of the streaming query.

In [ ]:
import json

if streaming_query.isActive:
    progress = streaming_query.lastProgress
    if progress:
        print("Last batch progress:")
        print(json.dumps(progress, indent=2, default=str))
    else:
        print("Stream is active — no batches processed yet. Waiting...")

    status = streaming_query.status
    print(f"\nStatus: {json.dumps(status, indent=2)}")

    print(f"\nCumulative stats:")
    print(f"  Batches processed : {_batch_stats['batches_processed']}")
    print(f"  Total Silver rows : {_batch_stats['total_silver']:,}")
    print(f"  Total Quarantine  : {_batch_stats['total_quarantine']:,}")
else:
    print("⚠ Stream is NOT active.")
    if streaming_query.exception():
        print(f"Exception: {streaming_query.exception()}")

## Stop Stream (Run when done)

In [ ]:
# ⚠ Run this cell ONLY when you want to stop the streaming query.
# The checkpoint ensures the next run resumes from where it left off.

# streaming_query.stop()
# print("✓ Streaming query stopped.")
# print(f"  Final stats:")
# print(f"    Batches : {_batch_stats['batches_processed']}")
# print(f"    Silver  : {_batch_stats['total_silver']:,} rows")
# print(f"    Quarant : {_batch_stats['total_quarantine']:,} rows")

print("Uncomment the lines above to stop the stream.")